Load the word dictionary and a pre-compiled patterns grid. Initialize a game engine with those values.

In [ ]:
from matplotlib import pyplot as plt
from cache import load_words, load_patterns
from engine import Engine

words = load_words()
patterns = load_patterns()
engine = Engine(words, patterns)

plt.rcParams['figure.figsize'] = (9, 6)
plt.rcParams['figure.dpi'] = 200

Given a list of solutions, use the engine as a bot to simulate a Wordle game played according to its strategy. Record the entropy history after each guess.

In [ ]:
from typing import Dict, List
from tqdm import tqdm

# PNW, anyone?
SOLUTIONS = [
    'water',
    'climb',
    'rainy',
    'talus',
    'coast',
    'cedar',
    'pines',
    'foggy',
    'orcas',
    'stars',
]

history: Dict[str, List[float]] = {}
for solution in tqdm(SOLUTIONS):
    engine.simulate(solution)
    history[solution] = engine.entropy_hist


Graph the entropy history to visualize how the engine iteratively prunes the possibilities space after each round.

In [ ]:
import numpy as np

max_guesses = max(len(sol) for sol in history.values())

plt.figure(figsize=(9, 6), dpi=200)
plt.title('Simulated Wordle Games')
plt.xlabel('Round')
plt.ylabel('Entropy (bits)')
plt.grid(True)

for word, entrops in history.items():
    plt.plot(entrops, label=word)

plt.xticks(ticks=np.arange(max_guesses))
plt.legend(loc='best')
plt.savefig('resources/simulation_graph.png')
plt.show()

Examine the expected information associated with an initial guess. Start by counting the occurences of each square pattern.

In [ ]:
import pandas as pd

EXAMPLE_WORD = 'fjord'

distribution = patterns[words == EXAMPLE_WORD, :]
squares, count = np.unique(distribution, return_counts=True)
df = pd.DataFrame(data=count, columns=['matches'],
                  index=pd.Index(squares, name='squares'))
df.sort_values('matches', ascending=False, inplace=True)
df

Plotting probability against information, we see that they are inversely correlated.

In [ ]:
probabilities = (df.matches / df.matches.sum()).to_numpy()
informations = -np.log2(probabilities)

fig, ax1 = plt.subplots()

ax1.set_title(f'Distribution for {EXAMPLE_WORD}')
ax1.set_xlabel('Sorted Index')
ax1.set_ylabel('Probability')
lns1 = ax1.plot(probabilities, label='probability', color='tab:blue')

ax2 = ax1.twinx()
ax2.set_ylabel('Information (bits)')
lns2 = ax2.plot(informations, label='information', color='tab:red')

lns = lns1 + lns2
labs: List[str] = [lnx.get_label() for lnx in lns]  # type: ignore
ax1.legend(lns, labs, loc='upper center')

fig.tight_layout()
plt.savefig('resources/information_graph.png')
plt.show()

Compute the entropy and assert that it matches the value given by the `scipy.stats` function.

In [ ]:
from scipy.stats import entropy

computed_entropy = np.dot(probabilities, informations)
assert abs(computed_entropy - entropy(df, base=2)) < 1e-5
computed_entropy

Use the engine to run this calculation across all words in the dictionary to determine the best initial guess.

In [ ]:
engine.reset()
words_ent, entropies = engine.ranker.informative_guesses()
ent_df = pd.DataFrame(index=pd.Index(words_ent, name='word'),
                         data=entropies, columns=['entropy'])
words_freq, frequency = engine.ranker.likely_solutions()
freq_df = pd.DataFrame(index=pd.Index(words_freq, name='word'),
                       data=frequency, columns=['log_freq'])
init_df = pd.merge(ent_df, freq_df, left_index=True, right_index=True)

init_df.sort_index().to_csv('resources/initial_data.csv')
init_df.sort_values(by='entropy', ascending=False)